In [1]:
# =============================================================================
# SPAM EMAIL CLASSIFIER — IMPROVED v2
# All changes are marked with [FIX], [IMPROVE], or [NEW]
# Copy each section into the corresponding notebook cell
# =============================================================================


# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — IMPORTS  (no changes needed here)
# ─────────────────────────────────────────────────────────────────────────────
import re
import joblib
import numpy as np
import pandas as pd
import nltk
import seaborn as sns
import matplotlib.pyplot as plt

from nltk.corpus import stopwords, wordnet
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV        # [NEW] needed for SVM probabilities
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from scipy.sparse import hstack, csr_matrix

from xgboost import XGBClassifier

DATA_PATH = "../spam_email_dataset.csv"


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — LOAD & EDA  (no changes needed here)
# ─────────────────────────────────────────────────────────────────────────────
data = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {data.shape}")
print(f"Duplicates: {data.duplicated().sum()}")
print(f"Missing values:\n{data.isna().sum()}")
print(f"\nLabel distribution:\n{data['label'].value_counts()}")
data.head()


Dataset shape: (10000, 20)
Duplicates: 0
Missing values:
email_id                   0
subject                    0
email_text                 0
num_words                  0
num_characters             0
num_exclamation_marks      0
num_links                  0
has_suspicious_link        0
num_attachments            0
has_attachment             0
sender_email               0
sender_domain              0
sender_reputation_score    0
email_hour                 0
email_day_of_week          0
is_weekend                 0
num_recipients             0
contains_money_terms       0
contains_urgency_terms     0
label                      0
dtype: int64

Label distribution:
label
0    6005
1    3995
Name: count, dtype: int64


,email_id,subject,email_text,num_words,num_characters,num_exclamation_marks,num_links,has_suspicious_link,num_attachments,has_attachment,sender_email,sender_domain,sender_reputation_score,email_hour,email_day_of_week,is_weekend,num_recipients,contains_money_terms,contains_urgency_terms,label
0,0,Weekly Report,budget review - Statement our I claim world st...,19,114,0,2,0,2,1,lctvdzm@outlook.com,outlook.com,0.66,19,3,0,23,0,0,0
1,1,Project Update,team sync - President series today already. In...,18,114,0,7,0,0,0,pxyldmi@company.com,company.com,0.95,4,4,0,16,1,0,0
2,2,🔥WIN BIG NOW!!,win free urgent offer limited limited urgent u...,19,126,0,4,1,1,1,atvanls@unknownmail.cc,unknownmail.cc,0.68,3,0,0,10,1,1,1
3,3,🔥WIN BIG NOW!!,guarantee click now cash offer click now guara...,16,101,0,7,1,1,1,qalxcnf@chealdealz.xyz,chealdealz.xyz,0.69,19,5,1,25,1,1,1
4,4,Meeting Reminder,team sync - Significant property hotel not add...,18,111,0,7,1,2,1,xoiccxl@yahoo.com,yahoo.com,0.67,4,5,1,8,0,0,0


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — [NEW] FEATURE LEAKAGE AUDIT
# Run this BEFORE training to understand how much the pre-computed metadata
# columns already "know" the answer. If metadata-only accuracy is very high
# (e.g. >90%), the dataset has leakage and reported model accuracy is inflated.
# ─────────────────────────────────────────────────────────────────────────────
from sklearn.ensemble import GradientBoostingClassifier

leakage_cols = [
    'contains_money_terms', 'contains_urgency_terms',
    'has_suspicious_link', 'sender_reputation_score',
    'num_recipients'
]
X_leak = data[leakage_cols]
y_leak = data['label']

# Quick 80/20 check — no tuning, just a signal
from sklearn.model_selection import train_test_split as tts
Xl_tr, Xl_te, yl_tr, yl_te = tts(X_leak, y_leak, test_size=0.2, random_state=42, stratify=y_leak)
leak_model = GradientBoostingClassifier(n_estimators=50, random_state=42)
leak_model.fit(Xl_tr, yl_tr)
leak_acc = accuracy_score(yl_te, leak_model.predict(Xl_te))

print("=" * 55)
print(f"  Metadata-ONLY accuracy (leakage proxy): {leak_acc:.4f}")
if leak_acc > 0.90:
    print("  ⚠  HIGH LEAKAGE — metadata columns already encode")
    print("     the label. Take overall model accuracy with caution.")
elif leak_acc > 0.75:
    print("  ⚠  MODERATE LEAKAGE — some inflation is likely.")
else:
    print("  ✓  LOW LEAKAGE — metadata adds signal but doesn't")
    print("     trivially predict the label.")
print("=" * 55)

  Metadata-ONLY accuracy (leakage proxy): 0.8515
  ⚠  MODERATE LEAKAGE — some inflation is likely.


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — TEXT PREPROCESSING  (no changes needed here)
# ─────────────────────────────────────────────────────────────────────────────
data.drop(
    columns=["email_id", "sender_email", "has_attachment",
             "num_characters", "num_exclamation_marks"],
    inplace=True
)

for pkg in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger_eng', 'omw-1.4']:
    nltk.download(pkg, quiet=True)

lemmatizer = WordNetLemmatizer()
stop_words  = set(stopwords.words('english'))

data['full_text'] = data['subject'].fillna('') + " " + data['email_text'].fillna('')


def get_wordnet_pos(treebank_tag: str) -> str:
    mapping = {'J': wordnet.ADJ, 'V': wordnet.VERB,
               'N': wordnet.NOUN, 'R': wordnet.ADV}
    return mapping.get(treebank_tag[0], wordnet.NOUN)


def preprocess_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    words = text.split()
    pos_tags = nltk.pos_tag(words)
    return ' '.join(
        lemmatizer.lemmatize(word, pos=get_wordnet_pos(tag))
        for word, tag in pos_tags
        if word not in stop_words and len(word) > 1
    )


print("Processing texts… (takes a few minutes)")
data['clean_text'] = data['full_text'].apply(preprocess_text)
data.drop(columns=['subject', 'email_text', 'full_text'], inplace=True)
print("Done. Remaining columns:", data.columns.tolist())

Processing texts… (takes a few minutes)
Done. Remaining columns: ['num_words', 'num_links', 'has_suspicious_link', 'num_attachments', 'sender_domain', 'sender_reputation_score', 'email_hour', 'email_day_of_week', 'is_weekend', 'num_recipients', 'contains_money_terms', 'contains_urgency_terms', 'label', 'clean_text']


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — FEATURE ENGINEERING  (no changes needed here)
# ─────────────────────────────────────────────────────────────────────────────
X_text    = data['clean_text']
X_numeric = data.drop(columns=['clean_text', 'label'])
y         = data['label']

print(f"Numeric features ({X_numeric.shape[1]}): {X_numeric.columns.tolist()}")

Numeric features (12): ['num_words', 'num_links', 'has_suspicious_link', 'num_attachments', 'sender_domain', 'sender_reputation_score', 'email_hour', 'email_day_of_week', 'is_weekend', 'num_recipients', 'contains_money_terms', 'contains_urgency_terms']


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — SPLIT  (no changes needed here)
# ─────────────────────────────────────────────────────────────────────────────
X_text_train, X_text_test, X_num_train, X_num_test, y_train, y_test = train_test_split(
    X_text, X_numeric, y,
    test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {len(y_train)}  |  Test: {len(y_test)}")

Train: 8000  |  Test: 2000


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — ENCODE sender_domain  (no changes needed here)
# ─────────────────────────────────────────────────────────────────────────────
le = LabelEncoder()
X_num_train = X_num_train.copy()
X_num_test  = X_num_test.copy()

X_num_train['sender_domain_encoded'] = le.fit_transform(X_num_train['sender_domain'])
le_dict = dict(zip(le.classes_, le.transform(le.classes_)))
X_num_test['sender_domain_encoded']  = X_num_test['sender_domain'].apply(lambda x: le_dict.get(x, -1))

X_num_train.drop(columns=['sender_domain'], inplace=True)
X_num_test.drop(columns=['sender_domain'], inplace=True)

# [NEW] Save the exact ordered feature list so app.py can reproduce the same columns
NUMERIC_FEATURE_NAMES = X_num_train.columns.tolist()
print("Numeric feature order:", NUMERIC_FEATURE_NAMES)

Numeric feature order: ['num_words', 'num_links', 'has_suspicious_link', 'num_attachments', 'sender_reputation_score', 'email_hour', 'email_day_of_week', 'is_weekend', 'num_recipients', 'contains_money_terms', 'contains_urgency_terms', 'sender_domain_encoded']


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — VECTORISE + SCALE  (no changes needed here)
# ─────────────────────────────────────────────────────────────────────────────
tfidf = TfidfVectorizer(max_features=3000, ngram_range=(1, 1), min_df=5)
X_text_train_tfidf = tfidf.fit_transform(X_text_train)
X_text_test_tfidf  = tfidf.transform(X_text_test)

scaler = StandardScaler()
X_num_train_scaled = scaler.fit_transform(X_num_train)
X_num_test_scaled  = scaler.transform(X_num_test)

X_train_combined = csr_matrix(hstack((X_text_train_tfidf, X_num_train_scaled)))
X_test_combined  = csr_matrix(hstack((X_text_test_tfidf,  X_num_test_scaled)))

print(f"Train shape: {X_train_combined.shape}  |  Test shape: {X_test_combined.shape}")


Train shape: (8000, 900)  |  Test shape: (2000, 900)


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — LINEAR SVM
# [FIX] Wrap with CalibratedClassifierCV so the app gets real probabilities
#        instead of the useless "N/A (Linear SVM Boundary Metric)" label.
# ─────────────────────────────────────────────────────────────────────────────
print("--- Linear SVM ---")
base_svm = LinearSVC(random_state=42, max_iter=2000)
svm_model = CalibratedClassifierCV(base_svm, cv=5)   # [FIX] calibrated wrapper
svm_model.fit(X_train_combined, y_train)
svm_predictions = svm_model.predict(X_test_combined)
print(classification_report(y_test, svm_predictions))
print("Accuracy:", accuracy_score(y_test, svm_predictions))

# Cross-val on the base estimator (faster)
svm_cv = cross_val_score(LinearSVC(random_state=42, max_iter=2000),
                          X_train_combined, y_train, cv=5, scoring='f1')
print(f"Cross-Val F1 (SVM): {svm_cv.mean():.4f} ± {svm_cv.std():.4f}")

--- Linear SVM ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1201
           1       1.00      1.00      1.00       799

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

Accuracy: 1.0
Cross-Val F1 (SVM): 1.0000 ± 0.0000


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — XGBOOST  (no changes needed here)
# ─────────────────────────────────────────────────────────────────────────────
print("--- XGBoost ---")
xgb_model = XGBClassifier(
    n_estimators=100, learning_rate=0.1,
    random_state=42, eval_metric='logloss', tree_method='hist'
)
xgb_model.fit(X_train_combined, y_train)
xgb_predictions = xgb_model.predict(X_test_combined)
print(classification_report(y_test, xgb_predictions))
print("Accuracy:", accuracy_score(y_test, xgb_predictions))

xgb_cv = cross_val_score(
    XGBClassifier(n_estimators=100, learning_rate=0.1,
                  random_state=42, eval_metric='logloss', tree_method='hist'),
    X_train_combined, y_train, cv=5, scoring='f1'
)
print(f"Cross-Val F1 (XGBoost): {xgb_cv.mean():.4f} ± {xgb_cv.std():.4f}")

--- XGBoost ---
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1201
           1       1.00      1.00      1.00       799

    accuracy                           1.00      2000
   macro avg       1.00      1.00      1.00      2000
weighted avg       1.00      1.00      1.00      2000

Accuracy: 1.0
Cross-Val F1 (XGBoost): 0.9998 ± 0.0003


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — DEEP LEARNING  (no changes needed here)
# ─────────────────────────────────────────────────────────────────────────────
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (Layer, Input, Embedding, LSTM,
                                      Dense, Bidirectional, Dropout, Concatenate)
from tensorflow.keras.models import Model
import tensorflow.keras.backend as K

MAX_VOCAB_SIZE    = 5000
MAX_SEQUENCE_LENGTH = 150

dl_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
dl_tokenizer.fit_on_texts(X_text_train)

X_train_pad = pad_sequences(dl_tokenizer.texts_to_sequences(X_text_train),
                              maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')
X_test_pad  = pad_sequences(dl_tokenizer.texts_to_sequences(X_text_test),
                              maxlen=MAX_SEQUENCE_LENGTH, padding='post', truncating='post')


class AttentionLayer(Layer):
    def build(self, input_shape):
        # [FIX] Keras 3.x changed add_weight() signature — name must be keyword-only
        self.W = self.add_weight(name="att_weight", shape=(input_shape[-1], 1),
                                  initializer="normal", trainable=True)
        self.b = self.add_weight(name="att_bias",   shape=(input_shape[1],  1),
                                  initializer="zeros",  trainable=True)
        super().build(input_shape)

    def call(self, x):
        e = K.tanh(K.dot(x, self.W) + self.b)
        a = K.softmax(e, axis=1)
        return K.sum(x * a, axis=1)


EMBEDDING_DIM       = 64
NUMERIC_FEATURES_DIM = X_num_train_scaled.shape[1]

text_input    = Input(shape=(MAX_SEQUENCE_LENGTH,), name="text_input")
embedding     = Embedding(MAX_VOCAB_SIZE, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH)(text_input)
lstm          = Bidirectional(LSTM(64, return_sequences=True))(embedding)
attention_out = AttentionLayer()(lstm)
text_dense    = Dense(32, activation='relu')(attention_out)

numeric_input  = Input(shape=(NUMERIC_FEATURES_DIM,), name="numeric_input")
numeric_dense  = Dense(16, activation='relu')(numeric_input)

combined    = Concatenate()([text_dense, numeric_dense])
fc_layer    = Dense(32, activation='relu')(combined)
dropout     = Dropout(0.5)(fc_layer)
output_layer = Dense(1, activation='sigmoid')(dropout)

hybrid_model = Model(inputs=[text_input, numeric_input], outputs=output_layer)
hybrid_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
hybrid_model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ text_input          │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 150, 64)   │    320,000 │ text_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_1     │ (None, 150, 128)  │     66,048 │ embedding_1[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention_layer_1   │ (None, 128)       │        278 │ bidirectional_1[… │
│ (AttentionLayer)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ numeric_input       │ (None, 12)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      4,128 │ attention_layer_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 16)        │        208 │ numeric_input[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 48)        │          0 │ dense[0][0],      │
│ (Concatenate)       │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 32)        │      1,568 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 1)         │         33 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 392,263 (1.50 MB)

 Trainable params: 392,263 (1.50 MB)

 Non-trainable params: 0 (0.00 B)

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — TRAIN DEEP LEARNING  (no changes needed here)
# ─────────────────────────────────────────────────────────────────────────────
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=3, restore_best_weights=True)

history = hybrid_model.fit(
    [X_train_pad, X_num_train_scaled], y_train,
    epochs=10, batch_size=64, validation_split=0.1,
    callbacks=[early_stop], verbose=1
)

predictions_prob = hybrid_model.predict([X_test_pad, X_num_test_scaled])
dl_predictions   = (predictions_prob > 0.5).astype(int)
print(classification_report(y_test, dl_predictions))

Epoch 1/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 21s 130ms/step - accuracy: 0.8529 - loss: 0.3022 - val_accuracy: 1.0000 - val_loss: 7.1119e-06
Epoch 2/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 13s 112ms/step - accuracy: 0.9997 - loss: 0.0010 - val_accuracy: 1.0000 - val_loss: 1.8718e-07
Epoch 3/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 13s 116ms/step - accuracy: 1.0000 - loss: 3.4963e-04 - val_accuracy: 1.0000 - val_loss: 2.6543e-08
Epoch 4/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 14s 121ms/step - accuracy: 0.9999 - loss: 3.2901e-04 - val_accuracy: 1.0000 - val_loss: 6.0819e-09
Epoch 5/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 14s 124ms/step - accuracy: 1.0000 - loss: 1.4134e-04 - val_accuracy: 1.0000 - val_loss: 1.1909e-09
Epoch 6/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 14s 127ms/step - accuracy: 0.9999 - loss: 2.2531e-04 - val_accuracy: 1.0000 - val_loss: 2.5287e-10
Epoch 7/10
113/113 ━━━━━━━━━━━━━━━━━━━━ 14s 124ms/step - accuracy: 0.9997 - loss: 2.8108e-04 - val_accuracy: 1.0000 - val_loss: 1.4707e-10
Epoch 8/10
113/113 ━━━━━━━━━━━━━━━━

In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — SAVE ASSETS
# [FIX] Save model in the modern .keras format (not legacy .h5)
# [NEW] Save keras_tokenizer and numeric feature names so app.py
#       never has to guess the column order or rebuild the tokenizer
# ─────────────────────────────────────────────────────────────────────────────
joblib.dump(svm_model,            'spam_svm_model.joblib')
joblib.dump(xgb_model,            'spam_xgb_model.joblib')
joblib.dump(tfidf,                'tfidf_vectorizer.joblib')
joblib.dump(scaler,               'scaler.joblib')
joblib.dump(le,                   'label_encoder.joblib')
joblib.dump(dl_tokenizer,         'keras_tokenizer.joblib')         # [NEW] was missing
joblib.dump(NUMERIC_FEATURE_NAMES,'numeric_feature_names.joblib')  # [NEW] column order

# [FIX] Use .keras format instead of legacy .h5
hybrid_model.save('spam_hybrid_attention_model.keras')

print("All models and pipeline assets saved successfully.")
print("Files written:")
for f in ['spam_svm_model.joblib', 'spam_xgb_model.joblib',
          'tfidf_vectorizer.joblib', 'scaler.joblib', 'label_encoder.joblib',
          'keras_tokenizer.joblib', 'numeric_feature_names.joblib',
          'spam_hybrid_attention_model.keras']:
    print(f"  ✓  {f}")


All models and pipeline assets saved successfully.
Files written:
  ✓  spam_svm_model.joblib
  ✓  spam_xgb_model.joblib
  ✓  tfidf_vectorizer.joblib
  ✓  scaler.joblib
  ✓  label_encoder.joblib
  ✓  keras_tokenizer.joblib
  ✓  numeric_feature_names.joblib
  ✓  spam_hybrid_attention_model.keras
